---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# Neural Networks — LOWO GroupKFold (cv=10)

## Objective
Compare two neural network architectures for DT prediction:
1. **Deep MLP** `Input(4) → Dense(64) → Dense(32) → Dense(16) → Output(1)`
2. **1D-CNN** `Conv1D(64) → MaxPool → Conv1D(128) → MaxPool → Dense(64) → Dense(32) → Output(1)`

## Methodology
- Leave-One-Well-Out (LOWO) — 27 iterations
- GroupKFold (cv=10) for internal validation
- PyTorch with early stopping and ReduceLROnPlateau
- Learning curves saved per well for convergence diagnostics

---
## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import json
from sonic_ml_utils import plot_well_profile_and_hexabin, statistics

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('deep')

In [ ]:
# ── Global settings ──────────────────────────────────────────────────────────
RANDOM_SEED = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU not available — using CPU')

In [ ]:
# ── Directories ──────────────────────────────────────────────────────────────
MLP_DIR      = Path('../results/mlp')
MLP_METRICS  = MLP_DIR / 'metrics'
MLP_PREDS    = MLP_DIR / 'predictions'
MLP_FIGURES  = MLP_DIR / 'figures'
MLP_HISTORY  = MLP_DIR / 'history'

CNN_DIR      = Path('../results/cnn')
CNN_METRICS  = CNN_DIR / 'metrics'
CNN_PREDS    = CNN_DIR / 'predictions'
CNN_FIGURES  = CNN_DIR / 'figures'
CNN_HISTORY  = CNN_DIR / 'history'

COMPARISON_FIGURES = Path('../results/comparison/figures')

for d in [MLP_METRICS, MLP_PREDS, MLP_FIGURES, MLP_HISTORY,
          CNN_METRICS, CNN_PREDS, CNN_FIGURES, CNN_HISTORY,
          COMPARISON_FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

FEATURES = ['GR', 'RT90', 'RHOB', 'NPHI']
TARGET   = 'DT'
R2_THRESHOLD = 0.70


---
## 2. Neural Network Architectures

In [ ]:
class MLPRegressor(nn.Module):
    """
    Deep MLP for Sonic Log (DT) prediction
    Input(4) -> Dense(64) -> Dense(32) -> Dense(16) -> Output(1)
    BatchNorm + ReLU + Dropout on each hidden layer
    """
    def __init__(self, input_dim=4, dropout=0.2):
        super(MLPRegressor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),        nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 16),        nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

# Test
m = MLPRegressor()
print(f'MLP — parameters: {sum(p.numel() for p in m.parameters()):,}')
print(f'  Input (10, 4) → Output {m(torch.randn(10, 4)).shape}')

In [ ]:
class CNN1DRegressor(nn.Module):
    """
    1D-CNN for DT prediction using depth context
    Input(window_size, 4) -> Conv1D(64) -> MaxPool -> Conv1D(128) -> MaxPool
                          -> Flatten -> Dense(64) -> Dense(32) -> Output(1)
    """
    def __init__(self, input_dim=4, window_size=11, dropout=0.3):
        super(CNN1DRegressor, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(input_dim, 64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,       128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        conv_out = 128 * (window_size // 2 // 2)
        self.fc_layers = nn.Sequential(
            nn.Linear(conv_out, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),       nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)          # (batch, features, window)
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)      # flatten
        return self.fc_layers(x)

# Test
m = CNN1DRegressor()
print(f'CNN — parameters: {sum(p.numel() for p in m.parameters()):,}')
print(f'  Input (10, 11, 4) → Output {m(torch.randn(10, 11, 4)).shape}')

---
## 3. Helper Functions

In [ ]:
def create_sequences(X, y, window_size=11):
    """Creates sliding depth windows for CNN."""
    half = window_size // 2
    X_pad = np.pad(X, ((half, half), (0, 0)), mode='edge')
    X_seq = np.array([X_pad[i:i + window_size] for i in range(len(X))])
    return X_seq, y

# Test
Xs, ys = create_sequences(np.random.randn(100, 4), np.random.randn(100))
print(f'create_sequences: (100, 4) → {Xs.shape}')

In [ ]:
def train_model(model, train_loader, val_loader,
                epochs=200, lr=0.001, patience=20,
                device=DEVICE, verbose=False):
    """
    Trains with early stopping and ReduceLROnPlateau.
    Returns (model, history) where history = {'train_loss', 'val_loss', 'best_epoch'}.
    """
    model     = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=False
    )

    history = {'train_loss': [], 'val_loss': [], 'best_epoch': 0}

    best_val_loss    = float('inf')
    patience_counter = 0
    best_state       = None

    for epoch in range(epochs):
        # ── Training ─────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb).squeeze(), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                val_loss += criterion(model(Xb).squeeze(), yb).item()
        val_loss /= len(val_loader)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        scheduler.step(val_loss)

        # ── Early stopping ───────────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss            = val_loss
            patience_counter         = 0
            best_state               = {k: v.clone() for k, v in model.state_dict().items()}
            history['best_epoch']    = epoch + 1
        else:
            patience_counter += 1

        if verbose and (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1:3d}  train={train_loss:.4f}  val={val_loss:.4f}')

        if patience_counter >= patience:
            if verbose:
                print(f'  Early stop at epoch {epoch + 1}')
            break

    if best_state:
        model.load_state_dict(best_state)

    return model, history


def predict_model(model, X, device=DEVICE):
    """Inference in eval mode, without gradient computation."""
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X, dtype=torch.float32, device=device))
    return preds.cpu().numpy().squeeze()


print('train_model / predict_model defined.')

In [ ]:
def plot_learning_curves(histories, method_name, figures_dir,
                         n_cols=4, figsize_per_plot=(4.5, 3.2)):
    """
    Plots learning curves (train/val loss) for each well.

    Parameters
    ----------
    histories    : dict  {well_name: {'train_loss', 'val_loss', 'best_epoch'}}
    method_name  : str   'MLP' or 'CNN'
    figures_dir  : Path
    n_cols       : int   columns in the grid
    """
    wells  = sorted(histories.keys())
    n_wells = len(wells)
    n_rows  = (n_wells + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(figsize_per_plot[0] * n_cols, figsize_per_plot[1] * n_rows)
    )
    axes = axes.flatten()

    for i, well in enumerate(wells):
        ax  = axes[i]
        h   = histories[well]
        eps = range(1, len(h['train_loss']) + 1)

        ax.plot(eps, h['train_loss'], color='steelblue', lw=1.5, label='Training')
        ax.plot(eps, h['val_loss'],   color='tomato',    lw=1.5, label='Validation')
        ax.axvline(h['best_epoch'], color='seagreen', lw=1.2, linestyle='--',
                   label=f'Best: ep. {h["best_epoch"]}')

        ax.set_title(well, fontsize=8, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=7)
        ax.set_ylabel('MSE Loss', fontsize=7)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, loc='upper right')
        ax.grid(True, alpha=0.3)

    # Turn off empty axes
    for j in range(n_wells, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'{method_name} — Learning Curves per Well (LOWO)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_path = figures_dir / f'lowo_{method_name.lower()}_learning_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Curves saved: {save_path}')


def plot_learning_curves_summary(histories, method_name, figures_dir):
    """
    Summary plot: median + IQR of training and validation loss over epochs.
    Useful for visualizing the average model behavior across all wells.
    """
    max_epochs = max(len(h['train_loss']) for h in histories.values())

    # Align histories (pad with NaN for wells that stopped early)
    def pad(lst):
        return lst + [np.nan] * (max_epochs - len(lst))

    train_mat = np.array([pad(h['train_loss']) for h in histories.values()])
    val_mat   = np.array([pad(h['val_loss'])   for h in histories.values()])
    best_eps  = [h['best_epoch'] for h in histories.values()]

    eps = np.arange(1, max_epochs + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{method_name} — Learning Curves Summary (32 wells)',
                 fontsize=13, fontweight='bold')

    for ax, mat, label, color in [
        (axes[0], train_mat, 'Training',   'steelblue'),
        (axes[1], val_mat,   'Validation', 'tomato')
    ]:
        med  = np.nanmedian(mat, axis=0)
        q25  = np.nanpercentile(mat, 25, axis=0)
        q75  = np.nanpercentile(mat, 75, axis=0)

        ax.plot(eps, med, color=color, lw=2, label='Median')
        ax.fill_between(eps, q25, q75, color=color, alpha=0.25, label='IQR (25-75%)')

        # Distribution of early stopping epochs
        med_ep = int(np.median(best_eps))
        ax.axvline(med_ep, color='seagreen', lw=1.5, linestyle='--',
                   label=f'Median early stop: ep. {med_ep}')

        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('MSE Loss', fontsize=11)
        ax.set_title(f'{label} Loss', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = figures_dir / f'lowo_{method_name.lower()}_learning_curves_summary.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Summary saved: {save_path}')


print('Learning curve functions defined.')

---
## 4. Data Loading

In [ ]:
df = pd.read_csv('../data/processed/wells_iqr_with_lithology.csv')

print('DATASET')
print(f'  Samples  : {len(df):,}')
print(f'  Wells    : {df["Well_Name"].nunique()}')
print(f'  Features : {FEATURES}')
print(f'  Target   : {TARGET}')

In [ ]:
# ── GroupShuffleSplit validation (MLP and CNN) ───────────────────────────────
n_wells   = df['Well_Name'].nunique()
n_samples = len(df)
TEST_SIZE = 0.15  # mesmo valor usado no loop LOWO

print(f'Dataset   : {n_samples:,} samples | {n_wells} wells')
print(f'Test size : {TEST_SIZE:.0%} of training wells → internal validation')
print(f'Strategy  : for each LOWO test well, ~{round(n_wells * TEST_SIZE)}'
      f' of the {n_wells-1} training wells go to validation\n')

# Simulate the split for the first training well
example_well  = sorted(df['Well_Name'].unique())[0]
train_df_ex   = df[df['Well_Name'] != example_well]
X_ex          = train_df_ex[FEATURES].values
y_ex          = train_df_ex[TARGET].values
groups_ex     = train_df_ex['Well_Name'].values

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=1)
tr_idx, val_idx = next(gss.split(X_ex, y_ex, groups_ex))

train_wells_ex = set(groups_ex[tr_idx])
val_wells_ex   = set(groups_ex[val_idx])
overlap        = train_wells_ex & val_wells_ex

print(f'Example (LOWO test well = {example_well}):')
print(f'  Internal training : {len(train_wells_ex)} wells | {len(tr_idx):,} samples')
print(f'  Validation        : {len(val_wells_ex)} wells | {len(val_idx):,} samples')
print(f'  Validation wells  : {sorted(val_wells_ex)}')
print()

if overlap:
    print(f'[FAIL] Overlap detected: {overlap}')
else:
    print('[OK]  No well appears in both training and validation simultaneously.')
    print('[OK]  GroupShuffleSplit configured correctly.')

---
## 5. MLP — LOWO GroupKFold

In [ ]:
def lowo_mlp(df, features, target, n_splits=10,
             epochs=200, lr=0.001, patience=20, batch_size=256):
    """
    LOWO with GroupShuffleSplit (val=15%) for MLP.
    Returns a list of results with metrics, predictions, and training history.
    """
    results    = []
    well_names = sorted(df['Well_Name'].unique())

    print('=' * 80)
    print('DEEP MLP — LOWO GroupShuffleSplit (val=15%)')
    print(f'Architecture: Input(4) → Dense(64) → Dense(32) → Dense(16) → Output(1)')
    print(f'Epochs={epochs}  LR={lr}  Patience={patience}  BatchSize={batch_size}')
    print('=' * 80)

    t0 = time.time()

    for idx, test_well in enumerate(well_names, 1):
        t_well = time.time()
        print(f'[{idx:2d}/{len(well_names)}] {test_well:25s}', end=' | ')

        train_df = df[df['Well_Name'] != test_well].copy()
        test_df  = df[df['Well_Name'] == test_well].copy()

        X_train, y_train = train_df[features].values, train_df[target].values
        X_test,  y_test  = test_df[features].values,  test_df[target].values

        scaler          = StandardScaler()
        X_train_sc      = scaler.fit_transform(X_train)
        X_test_sc       = scaler.transform(X_test)

        # GroupShuffleSplit — randomly selects ~15% of wells for validation
        # Avoids the GroupKFold issue of always using the same fixed group,
        # which caused premature early stopping on geologically distinct wells.
        groups           = train_df['Well_Name'].values
        gss              = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=idx)
        tr_idx, val_idx  = next(gss.split(X_train_sc, y_train, groups))

        train_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_train_sc[tr_idx]),
                          torch.FloatTensor(y_train[tr_idx])),
            batch_size=batch_size, shuffle=True,
            num_workers=0, pin_memory=True, persistent_workers=False, drop_last=True
        )
        val_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_train_sc[val_idx]),
                          torch.FloatTensor(y_train[val_idx])),
            batch_size=batch_size, shuffle=False,
            num_workers=0, pin_memory=True, persistent_workers=False
        )

        model, history = train_model(
            MLPRegressor(input_dim=len(features), dropout=0.2),
            train_loader, val_loader,
            epochs=epochs, lr=lr, patience=patience, device=DEVICE
        )

        y_pred = predict_model(model, X_test_sc)
        r2     = r2_score(y_test, y_pred)
        rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
        mae    = mean_absolute_error(y_test, y_pred)

        time_well = time.time() - t_well
        print(f'R²={r2:.4f} | RMSE={rmse:.2f} | ép.={history["best_epoch"]} | {time_well:.1f}s')

        results.append({
            'Well_Name'     : test_well,
            'R2'            : r2,
            'RMSE'          : rmse,
            'MAE'           : mae,
            'N_samples_test': len(y_test),
            'Time_seconds'  : time_well,
            'predictions'   : y_pred,
            'actuals'       : y_test,
            'history'       : history,
        })

    print(f'LOWO MLP completed in {(time.time()-t0)/60:.1f} min')
    return results

In [ ]:
# MLP Training
mlp_results_list = lowo_mlp(
    df=df, features=FEATURES, target=TARGET,
    n_splits=10, epochs=200, lr=0.001, batch_size=256, patience=20
)

# Consolidate metrics
mlp_results = pd.DataFrame([{
    'Well_Name'     : r['Well_Name'],
    'R2'            : r['R2'],
    'RMSE'          : r['RMSE'],
    'MAE'           : r['MAE'],
    'N_samples_test': r['N_samples_test'],
    'Time_seconds'  : r['Time_seconds'],
    'best_epoch'    : r['history']['best_epoch'],
} for r in mlp_results_list])

# Consolidate predictions
preds_mlp = []
for r in mlp_results_list:
    dw = df[df['Well_Name'] == r['Well_Name']].copy().iloc[:len(r['predictions'])]
    dw['DT_real'] = r['actuals']
    dw['DT_pred'] = r['predictions']
    preds_mlp.append(dw[['DEPTH', 'Well_Name', 'DT_real', 'DT_pred'] + FEATURES])
df_preds_mlp = pd.concat(preds_mlp, ignore_index=True)

# Save training history (JSON)
mlp_histories = {r['Well_Name']: r['history'] for r in mlp_results_list}
with open(MLP_HISTORY / 'mlp_learning_histories.json', 'w') as f:
    json.dump(mlp_histories, f)

# Save metrics and predictions
mlp_results.to_csv(MLP_METRICS / 'mlp_lowo_results.csv', index=False)
df_preds_mlp.to_csv(MLP_PREDS / 'lowo_mlp_predictions.csv', index=False)

print(f'Metrics    : {MLP_METRICS}/mlp_lowo_results.csv')
print(f'Predictions: {MLP_PREDS}/lowo_mlp_predictions.csv')
print(f'History    : {MLP_HISTORY}/mlp_learning_histories.json')
print(f'\nMean R²    : {mlp_results["R2"].mean():.4f}')
print(f'Mean RMSE  : {mlp_results["RMSE"].mean():.2f}')
print(f'Median epochs: {mlp_results["best_epoch"].median():.0f}')

In [ ]:
# Cache — load if training already exists
mlp_results  = pd.read_csv(MLP_METRICS / 'mlp_lowo_results.csv')
df_preds_mlp = pd.read_csv(MLP_PREDS / 'lowo_mlp_predictions.csv')

with open(MLP_HISTORY / 'mlp_learning_histories.json') as f:
    mlp_histories = json.load(f)

print(f'MLP loaded from cache.')
print(f'  Metrics     : {mlp_results.shape}')
print(f'  Predictions : {df_preds_mlp.shape}')
print(f'  Histories   : {len(mlp_histories)} wells')

### 5.1 Learning Curves — MLP

In [ ]:
# Overall summary (median + IQR)
plot_learning_curves_summary(mlp_histories, 'MLP', MLP_FIGURES)

In [ ]:
# Grid with all wells
plot_learning_curves(mlp_histories, 'MLP', MLP_FIGURES, n_cols=4)

In [ ]:
# Distribution of early stopping epochs
best_epochs_mlp = [h['best_epoch'] for h in mlp_histories.values()]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(best_epochs_mlp, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(np.median(best_epochs_mlp), color='tomato', lw=2, linestyle='--',
           label=f'Median: {np.median(best_epochs_mlp):.0f} epochs')
ax.set_xlabel('Early stopping epoch', fontsize=11)
ax.set_ylabel('Frequency (wells)', fontsize=11)
ax.set_title('MLP — Distribution of Convergence Epochs (32 wells)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MLP_FIGURES / 'lowo_mlp_early_stop_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Epochs  — Min: {min(best_epochs_mlp)}  Max: {max(best_epochs_mlp)}  Median: {np.median(best_epochs_mlp):.0f}')

### 5.2 Statistical Analyses — MLP

In [ ]:
statistics.print_summary_table(mlp_results)

statistics.analyze_lowo_results(
    results_df=mlp_results,
    predictions_df=df_preds_mlp,
    algorithm_name='MLP'
)

# Problematic wells
prob_mlp = mlp_results[mlp_results['R2'] < R2_THRESHOLD]
good_mlp = mlp_results[mlp_results['R2'] >= R2_THRESHOLD]
print(f'\nGood wells (R² ≥ {R2_THRESHOLD})         : {len(good_mlp)} ({len(good_mlp)/len(mlp_results)*100:.1f}%)  R²={good_mlp["R2"].mean():.4f}')
print(f'Problematic wells (R² < {R2_THRESHOLD})   : {len(prob_mlp)} ({len(prob_mlp)/len(mlp_results)*100:.1f}%)')
for _, row in prob_mlp.sort_values('R2').iterrows():
    print(f'  {row["Well_Name"]:30s} R²={row["R2"]:.4f}  RMSE={row["RMSE"]:.2f}')

# Residuals
yt_mlp = df_preds_mlp['DT_real'].values
yp_mlp = df_preds_mlp['DT_pred'].values
res    = statistics.residual_analysis(yt_mlp, yp_mlp)
print(f'\nResiduals  Mean={res["mean"]:.4f}  Std={res["std"]:.4f}  Skew={res["skewness"]:.4f}  Kurt={res["kurtosis"]:.4f}')

# Error by range
print('\nError by DT range:')
print(statistics.error_by_range(yt_mlp, yp_mlp, n_bins=10).to_string(index=False))

# CI
for metric, vals in [('R²', mlp_results['R2'].values),
                      ('RMSE', mlp_results['RMSE'].values),
                      ('MAE',  mlp_results['MAE'].values)]:
    ci = statistics.calculate_confidence_interval(vals, confidence=0.95)
    print(f'{metric:6s}: {ci["mean"]:.4f}  95% CI: [{ci["lower_bound"]:.4f}, {ci["upper_bound"]:.4f}]  ±{ci["margin_of_error"]:.4f}')

In [ ]:
# Bootstrap
bootstrap_mlp = statistics.bootstrap_confidence_interval(
    y_true=yt_mlp, y_pred=yp_mlp, metric_func=r2_score,
    n_bootstrap=1000, confidence=0.95
)
print(f'Bootstrap R²: {bootstrap_mlp["mean"]:.4f}  95% CI: [{bootstrap_mlp["lower_bound"]:.4f}, {bootstrap_mlp["upper_bound"]:.4f}]')

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(bootstrap_mlp['values'], bins=40, color='steelblue', alpha=0.75, edgecolor='white')
ax.axvline(bootstrap_mlp['mean'],        color='black',  lw=2,   linestyle='-',
           label=f'Mean: {bootstrap_mlp["mean"]:.4f}')
ax.axvline(bootstrap_mlp['lower_bound'], color='tomato', lw=1.5, linestyle='--',
           label=f'95% CI: [{bootstrap_mlp["lower_bound"]:.4f}, {bootstrap_mlp["upper_bound"]:.4f}]')
ax.axvline(bootstrap_mlp['upper_bound'], color='tomato', lw=1.5, linestyle='--')
ax.set_xlabel('R²', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Bootstrap R² — MLP LOWO (1000 iterations)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MLP_FIGURES / 'lowo_mlp_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Profile Visualizations — MLP

In [ ]:
# All wells
plot_well_profile_and_hexabin(
    predictions_df=df_preds_mlp,
    well_name=None,
    figsize=(10, 8),
    cmap='YlGnBu',
    gridsize=70,
    save_path=str(MLP_FIGURES / 'lowo_mlp_all_wells.png')
)


In [ ]:
# Top 5 best and worst wells
best_5  = mlp_results.nlargest(5, 'R2')
worst_5 = mlp_results.nsmallest(5, 'R2')

print('TOP 5 BEST:')
for _, row in best_5.iterrows():
    print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f}')

print('\nTOP 5 WORST:')
for _, row in worst_5.iterrows():
    print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f}')


---
## 6. 1D-CNN — LOWO GroupKFold

In [ ]:
def lowo_cnn(df, features, target, window_size=11, n_splits=10,
             epochs=200, lr=0.001, patience=20, batch_size=256):
    """
    LOWO with GroupShuffleSplit (val=15%) for 1D-CNN.
    Returns a list of results with metrics, predictions, and training history.
    """
    results    = []
    well_names = sorted(df['Well_Name'].unique())

    print('=' * 80)
    print('1D-CNN — LOWO GroupShuffleSplit (val=15%)')
    print(f'Architecture: Conv1D(64) → MaxPool → Conv1D(128) → MaxPool → Dense(64) → Dense(32) → Output(1)')
    print(f'Window={window_size}  Epochs={epochs}  LR={lr}  Patience={patience}  BatchSize={batch_size}')
    print('=' * 80)

    t0 = time.time()

    for idx, test_well in enumerate(well_names, 1):
        t_well = time.time()
        print(f'[{idx:2d}/{len(well_names)}] {test_well:25s}', end=' | ')

        train_df = df[df['Well_Name'] != test_well].copy()
        test_df  = df[df['Well_Name'] == test_well].copy()

        X_train, y_train = train_df[features].values, train_df[target].values
        X_test,  y_test  = test_df[features].values,  test_df[target].values

        scaler      = StandardScaler()
        X_train_sc  = scaler.fit_transform(X_train)
        X_test_sc   = scaler.transform(X_test)

        # Sequences for CNN
        X_train_seq, y_train_seq = create_sequences(X_train_sc, y_train, window_size)
        X_test_seq,  y_test_seq  = create_sequences(X_test_sc,  y_test,  window_size)

        # GroupShuffleSplit — randomly selects ~15% of wells for validation
        # Avoids the GroupKFold issue of always using the same fixed group,
        # which caused premature early stopping on geologically distinct wells.
        groups          = train_df['Well_Name'].values
        gss             = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=idx)
        tr_idx, val_idx = next(gss.split(X_train_seq, y_train_seq, groups))

        train_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_train_seq[tr_idx]),
                          torch.FloatTensor(y_train_seq[tr_idx])),
            batch_size=batch_size, shuffle=True,
            num_workers=0, pin_memory=True, persistent_workers=False, drop_last=True
        )
        val_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_train_seq[val_idx]),
                          torch.FloatTensor(y_train_seq[val_idx])),
            batch_size=batch_size, shuffle=False,
            num_workers=0, pin_memory=True, persistent_workers=False
        )

        model, history = train_model(
            CNN1DRegressor(input_dim=len(features), window_size=window_size, dropout=0.3),
            train_loader, val_loader,
            epochs=epochs, lr=lr, patience=patience, device=DEVICE
        )

        y_pred = predict_model(model, X_test_seq)
        r2     = r2_score(y_test_seq, y_pred)
        rmse   = np.sqrt(mean_squared_error(y_test_seq, y_pred))
        mae    = mean_absolute_error(y_test_seq, y_pred)

        tempo_poco = time.time() - t_well
        print(f'R²={r2:.4f} | RMSE={rmse:.2f} | ép.={history["best_epoch"]} | {tempo_poco:.1f}s')

        results.append({
            'Well_Name'     : test_well,
            'R2'            : r2,
            'RMSE'          : rmse,
            'MAE'           : mae,
            'N_samples_test': len(y_test_seq),
            'Time_seconds'  : tempo_poco,
            'predictions'   : y_pred,
            'actuals'       : y_test_seq,
            'history'       : history,
        })

    print('=' * 80)
    print(f'LOWO CNN completed in {(time.time()-t0)/60:.1f} min')
    print('=' * 80)
    return results

In [ ]:
# CNN Training
cnn_results_list = lowo_cnn(
    df=df, features=FEATURES, target=TARGET,
    window_size=11, n_splits=10, epochs=200, lr=0.001, batch_size=256, patience=20
)

# Consolidate metrics
cnn_results = pd.DataFrame([{
    'Well_Name'     : r['Well_Name'],
    'R2'            : r['R2'],
    'RMSE'          : r['RMSE'],
    'MAE'           : r['MAE'],
    'N_samples_test': r['N_samples_test'],
    'Time_seconds'  : r['Time_seconds'],
    'best_epoch'    : r['history']['best_epoch'],
} for r in cnn_results_list])

# Consolidate predictions
preds_cnn = []
for r in cnn_results_list:
    dw = df[df['Well_Name'] == r['Well_Name']].copy().iloc[:len(r['predictions'])]
    dw['DT_real'] = r['actuals']
    dw['DT_pred'] = r['predictions']
    preds_cnn.append(dw[['DEPTH', 'Well_Name', 'DT_real', 'DT_pred'] + FEATURES])
df_preds_cnn = pd.concat(preds_cnn, ignore_index=True)

# Save history
cnn_histories = {r['Well_Name']: r['history'] for r in cnn_results_list}
with open(CNN_HISTORY / 'cnn_learning_histories.json', 'w') as f:
    json.dump(cnn_histories, f)

# Save metrics and predictions
cnn_results.to_csv(CNN_METRICS / 'cnn1d_lowo_results.csv', index=False)
df_preds_cnn.to_csv(CNN_PREDS / 'lowo_cnn_predictions.csv', index=False)

print(f'Metrics    : {CNN_METRICS}/cnn1d_lowo_results.csv')
print(f'Predictions: {CNN_PREDS}/lowo_cnn_predictions.csv')
print(f'History    : {CNN_HISTORY}/cnn_learning_histories.json')
print(f'\nMean R²    : {cnn_results["R2"].mean():.4f}')
print(f'Mean RMSE  : {cnn_results["RMSE"].mean():.2f}')
print(f'Median epochs: {cnn_results["best_epoch"].median():.0f}')

In [ ]:
# Cache — load if training already exists
cnn_results  = pd.read_csv(CNN_METRICS / 'cnn1d_lowo_results.csv')
df_preds_cnn = pd.read_csv(CNN_PREDS / 'lowo_cnn_predictions.csv')

with open(CNN_HISTORY / 'cnn_learning_histories.json') as f:
    cnn_histories = json.load(f)

print(f'CNN loaded from cache.')
print(f'  Metrics     : {cnn_results.shape}')
print(f'  Predictions : {df_preds_cnn.shape}')
print(f'  Histories   : {len(cnn_histories)} wells')

### 6.1 Learning Curves — 1D-CNN

In [ ]:
plot_learning_curves_summary(cnn_histories, 'CNN', CNN_FIGURES)

In [ ]:
plot_learning_curves(cnn_histories, 'CNN', CNN_FIGURES, n_cols=4)

In [ ]:
best_epochs_cnn = [h['best_epoch'] for h in cnn_histories.values()]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(best_epochs_cnn, bins=20, color='coral', edgecolor='white', alpha=0.85)
ax.axvline(np.median(best_epochs_cnn), color='steelblue', lw=2, linestyle='--',
           label=f'Median: {np.median(best_epochs_cnn):.0f} epochs')
ax.set_xlabel('Early stopping epoch', fontsize=11)
ax.set_ylabel('Frequency (wells)', fontsize=11)
ax.set_title('1D-CNN — Distribution of Convergence Epochs (32 wells)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CNN_FIGURES / 'lowo_cnn_early_stop_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Epochs  — Min: {min(best_epochs_cnn)}  Max: {max(best_epochs_cnn)}  Median: {np.median(best_epochs_cnn):.0f}')

### 6.2 Statistical Analyses — 1D-CNN

In [ ]:
statistics.print_summary_table(cnn_results)

statistics.analyze_lowo_results(
    results_df=cnn_results,
    predictions_df=df_preds_cnn,
    algorithm_name='CNN-1D'
)

prob_cnn = cnn_results[cnn_results['R2'] < R2_THRESHOLD]
good_cnn = cnn_results[cnn_results['R2'] >= R2_THRESHOLD]
print(f'\nGood wells (R² ≥ {R2_THRESHOLD})         : {len(good_cnn)} ({len(good_cnn)/len(cnn_results)*100:.1f}%)  R²={good_cnn["R2"].mean():.4f}')
print(f'Problematic wells (R² < {R2_THRESHOLD})   : {len(prob_cnn)} ({len(prob_cnn)/len(cnn_results)*100:.1f}%)')
for _, row in prob_cnn.sort_values('R2').iterrows():
    print(f'  {row["Well_Name"]:30s} R²={row["R2"]:.4f}  RMSE={row["RMSE"]:.2f}')

yt_cnn = df_preds_cnn['DT_real'].values
yp_cnn = df_preds_cnn['DT_pred'].values
res    = statistics.residual_analysis(yt_cnn, yp_cnn)
print(f'\nResiduals  Mean={res["mean"]:.4f}  Std={res["std"]:.4f}  Skew={res["skewness"]:.4f}  Kurt={res["kurtosis"]:.4f}')

print('\nError by DT range:')
print(statistics.error_by_range(yt_cnn, yp_cnn, n_bins=10).to_string(index=False))

for metric, vals in [('R²', cnn_results['R2'].values),
                      ('RMSE', cnn_results['RMSE'].values),
                      ('MAE',  cnn_results['MAE'].values)]:
    ci = statistics.calculate_confidence_interval(vals, confidence=0.95)
    print(f'{metric:6s}: {ci["mean"]:.4f}  95% CI: [{ci["lower_bound"]:.4f}, {ci["upper_bound"]:.4f}]  ±{ci["margin_of_error"]:.4f}')

In [ ]:
bootstrap_cnn = statistics.bootstrap_confidence_interval(
    y_true=yt_cnn, y_pred=yp_cnn, metric_func=r2_score,
    n_bootstrap=1000, confidence=0.95
)
print(f'Bootstrap R²: {bootstrap_cnn["mean"]:.4f}  95% CI: [{bootstrap_cnn["lower_bound"]:.4f}, {bootstrap_cnn["upper_bound"]:.4f}]')

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(bootstrap_cnn['values'], bins=40, color='coral', alpha=0.75, edgecolor='white')
ax.axvline(bootstrap_cnn['mean'],        color='black',  lw=2,   linestyle='-',
           label=f'Mean: {bootstrap_cnn["mean"]:.4f}')
ax.axvline(bootstrap_cnn['lower_bound'], color='tomato', lw=1.5, linestyle='--',
           label=f'95% CI: [{bootstrap_cnn["lower_bound"]:.4f}, {bootstrap_cnn["upper_bound"]:.4f}]')
ax.axvline(bootstrap_cnn['upper_bound'], color='tomato', lw=1.5, linestyle='--')
ax.set_xlabel('R²', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Bootstrap R² — 1D-CNN LOWO (1000 iterations)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CNN_FIGURES / 'lowo_cnn_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Profile Visualizations — 1D-CNN

In [ ]:
# All wells
plot_well_profile_and_hexabin(
    predictions_df=df_preds_cnn,
    well_name=None,
    figsize=(10, 8),
    cmap='YlGnBu',
    gridsize=70,
    save_path=str(CNN_FIGURES / 'lowo_cnn_all_wells.png')
)

In [ ]:
# Top 5 best and worst wells
best_5  = cnn_results.nlargest(5, 'R2')
worst_5 = cnn_results.nsmallest(5, 'R2')

print('TOP 5 BEST:')
for _, row in best_5.iterrows():
    print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f}')

print('\nTOP 5 WORST:')
for _, row in worst_5.iterrows():
    print(f'  {row["Well_Name"]:28s} R²={row["R2"]:.4f}')


## 7. MLP vs 1D-CNN Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metrics': ['Mean R²', 'Median R²', 'R² std', 'Mean RMSE', 'Mean MAE',
                'Problematic wells (R²<0.50)', 'Median epochs'],
    'MLP': [
        f"{mlp_results['R2'].mean():.4f}",
        f"{mlp_results['R2'].median():.4f}",
        f"{mlp_results['R2'].std():.4f}",
        f"{mlp_results['RMSE'].mean():.2f}",
        f"{mlp_results['MAE'].mean():.2f}",
        f"{(mlp_results['R2'] < 0.50).sum()}",
        f"{mlp_results['best_epoch'].median():.0f}" if 'best_epoch' in mlp_results.columns else 'N/A',
    ],
    '1D-CNN': [
        f"{cnn_results['R2'].mean():.4f}",
        f"{cnn_results['R2'].median():.4f}",
        f"{cnn_results['R2'].std():.4f}",
        f"{cnn_results['RMSE'].mean():.2f}",
        f"{cnn_results['MAE'].mean():.2f}",
        f"{(cnn_results['R2'] < 0.50).sum()}",
        f"{cnn_results['best_epoch'].median():.0f}" if 'best_epoch' in cnn_results.columns else 'N/A',
    ]
})
print(comparison.to_string(index=False))

delta_r2   = cnn_results['R2'].mean() - mlp_results['R2'].mean()
delta_rmse = cnn_results['RMSE'].mean() - mlp_results['RMSE'].mean()
print(f'\nΔ R²   (CNN − MLP): {delta_r2:+.4f}')
print(f'Δ RMSE (CNN − MLP): {delta_rmse:+.2f} µs/ft')

if delta_r2 > 0.01:
    print('\n CNN is SIGNIFICANTLY BETTER — depth context adds value')
elif delta_r2 < -0.01:
    print('\n  MLP is better — depth context does not help')
else:
    print('\n≈ Tied (Δ R² < 0.01)')

In [ ]:
# Comparative boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd.DataFrame({'MLP': mlp_results['R2'].values,
              '1D-CNN': cnn_results['R2'].values}).boxplot(ax=axes[0])
axes[0].axhline(0.50, color='r', ls='--', alpha=0.5, label='Problematic threshold')
axes[0].axhline(0.80, color='g', ls='--', alpha=0.5, label='Excellent threshold')
axes[0].set_title('R² — MLP vs 1D-CNN', fontsize=13, fontweight='bold')
axes[0].set_ylabel('R²', fontsize=12)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

pd.DataFrame({'MLP': mlp_results['RMSE'].values,
              '1D-CNN': cnn_results['RMSE'].values}).boxplot(ax=axes[1])
axes[1].set_title('RMSE — MLP vs 1D-CNN', fontsize=13, fontweight='bold')
axes[1].set_ylabel('RMSE (µs/ft)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(COMPARISON_FIGURES / 'nn_comparison_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# R² scatter per well
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(mlp_results['R2'], cnn_results['R2'], s=100, alpha=0.6, edgecolors='black')
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='MLP = CNN')
for x, y, well in zip(mlp_results['R2'], cnn_results['R2'], mlp_results['Well_Name']):
    if abs(x - y) > 0.08:
        ax.annotate(well, (x, y), fontsize=6, alpha=0.7,
                    xytext=(4, 4), textcoords='offset points')
ax.axhline(0.5, color='orange', ls='--', alpha=0.3)
ax.axvline(0.5, color='orange', ls='--', alpha=0.3)
ax.axhline(0.8, color='green',  ls='--', alpha=0.3)
ax.axvline(0.8, color='green',  ls='--', alpha=0.3)
ax.set_xlabel('R² MLP', fontsize=12); ax.set_ylabel('R² 1D-CNN', fontsize=12)
ax.set_title('R² per Well: MLP vs 1D-CNN', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(COMPARISON_FIGURES / 'nn_scatter_well_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Learning curves MLP vs CNN — side-by-side summary
max_ep_mlp = max(len(h['train_loss']) for h in mlp_histories.values())
max_ep_cnn = max(len(h['train_loss']) for h in cnn_histories.values())

def pad_arr(lst, max_len):
    return lst + [np.nan] * (max_len - len(lst))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Learning Curves — MLP vs 1D-CNN (median over 27 wells)',
             fontsize=13, fontweight='bold')

for ax, histories, method, color_t, color_v, max_ep in [
    (axes[0], mlp_histories, 'MLP',    'steelblue', 'tomato',  max_ep_mlp),
    (axes[1], cnn_histories, '1D-CNN', 'darkorange', 'purple', max_ep_cnn),
]:
    train_mat = np.array([pad_arr(h['train_loss'], max_ep) for h in histories.values()])
    val_mat   = np.array([pad_arr(h['val_loss'],   max_ep) for h in histories.values()])
    eps       = np.arange(1, max_ep + 1)

    for mat, label, color in [(train_mat, 'Training', color_t), (val_mat, 'Validation', color_v)]:
        med = np.nanmedian(mat, axis=0)
        q25 = np.nanpercentile(mat, 25, axis=0)
        q75 = np.nanpercentile(mat, 75, axis=0)
        ax.plot(eps, med, color=color, lw=2, label=label)
        ax.fill_between(eps, q25, q75, color=color, alpha=0.2)

    best_eps = [h['best_epoch'] for h in histories.values()]
    ax.axvline(int(np.median(best_eps)), color='seagreen', lw=1.5, ls='--',
               label=f'Median early stop: ep. {int(np.median(best_eps))}')

    ax.set_title(method, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('MSE Loss', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(COMPARISON_FIGURES / 'nn_learning_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()